# DS2/GSE141383 — dataset-level quality control

This notebook reproduces the QC sequence recorded in the original
GSE141383 notebook while removing unrelated scVI/GSE154795 code and
machine-specific paths.

## Filtering sequence

1. calculate QC metrics from raw integer counts;
2. convert the feature axis from versioned Ensembl identifiers to gene
   symbols while retaining the original Ensembl identifiers in
   `adata.var["ensembl_id_version"]`;
3. retain cells satisfying all of:
   - detected genes > 300;
   - detected genes < 4,500;
   - total counts > 600;
   - total counts < 12,000;
   - mitochondrial fraction < 3%;
4. apply the final total-count ceiling of 9,000;
5. retain genes detected in at least 10 final cells.

Raw counts remain in both `adata.X` and `adata.layers["counts"]`.
No normalization or log transformation is performed.

## Expected output

- 21,391 cells
- 25,835 genes

Input:
`data/interim/DS2_GSE141383/GSE141383_merged.h5ad`

Output:
`data/processed/DS2_GSE141383/GSE141383_postQC.h5ad`

In [1]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)

Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
scanpy: 1.11.5
pandas: 2.3.3


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_18732\3006837562.py:14: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE141383_MERGED_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS2_GSE141383"
        / "GSE141383_merged.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE141383_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS2_GSE141383"
        / "GSE141383_postQC.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS2_GSE141383" / "00b_qc"
)
FIGURE_DIR = AUDIT_DIR / "figures"

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_INPUT_SHAPE = (29_415, 60_725)
EXPECTED_OUTPUT_SHAPE = (21_391, 25_835)
EXPECTED_AFTER_PRIMARY_CELL_FILTER = 21_500

MIN_GENES_EXCLUSIVE = 300
MAX_GENES_EXCLUSIVE = 4_500
MIN_COUNTS_EXCLUSIVE = 600
MAX_COUNTS_PRIMARY_EXCLUSIVE = 12_000
MAX_MT_PERCENT_EXCLUSIVE = 3.0
FINAL_MAX_COUNTS_INCLUSIVE = 9_000
FINAL_MIN_CELLS_PER_GENE = 10

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Merged input not found: {INPUT_H5AD}\n"
        "Run 00a_ds2_merge_per_sample_h5ad.ipynb first or set "
        "GSE141383_MERGED_H5AD."
    )

Input: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\interim\DS2_GSE141383\GSE141383_merged.h5ad
Output: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS2_GSE141383\GSE141383_postQC.h5ad


In [3]:
def matrix_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_counts(matrix, label: str):
    values = matrix_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} contains no nonzero count values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")


def save_current_figure(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close("all")

In [4]:
adata = sc.read_h5ad(INPUT_H5AD)

if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )
if "sample" not in adata.obs:
    raise KeyError("adata.obs['sample'] is required.")
if "gene_symbols" not in adata.var:
    if "gene_symbol" in adata.var:
        adata.var["gene_symbols"] = adata.var["gene_symbol"]
    else:
        raise KeyError(
            "adata.var must contain 'gene_symbols' or 'gene_symbol'."
        )
if not adata.obs_names.is_unique:
    raise AssertionError("Cell identifiers are not unique.")
if not adata.var_names.is_unique:
    raise AssertionError("Input Ensembl feature identifiers are not unique.")

assert_raw_counts(adata.X, "adata.X")

if sp.issparse(adata.X):
    adata.X = adata.X.tocsr()
adata.layers["counts"] = adata.X.copy()
assert_raw_counts(adata.layers["counts"], 'adata.layers["counts"]')

print(adata)

AnnData object with n_obs × n_vars = 29415 × 60725
    obs: 'cell_barcode', 'sample', 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset', 'source_file'
    var: 'gene_symbols'
    uns: 'build_timestamp', 'dataset_id', 'geo_accession', 'input_format', 'source_files'
    layers: 'counts'


In [5]:
import numpy as np
import pandas as pd
import scanpy as sc


# ------------------------------------------------------------------
# Validate the gene-symbol annotation column
# ------------------------------------------------------------------
GENE_SYMBOL_COLUMN = "gene_symbols"

if GENE_SYMBOL_COLUMN not in adata.var.columns:
    raise KeyError(
        f"Missing adata.var[{GENE_SYMBOL_COLUMN!r}]. "
        f"Available columns: {adata.var.columns.tolist()}"
    )


# Convert categorical annotations to pandas StringDtype before fillna.
gene_symbols = (
    adata.var[GENE_SYMBOL_COLUMN]
    .astype("string")
    .fillna("")
    .str.upper()
)


# ------------------------------------------------------------------
# Construct ordinary NumPy boolean masks
# ------------------------------------------------------------------
adata.var["mt"] = (
    gene_symbols
    .str.startswith("MT-", na=False)
    .to_numpy(dtype=bool)
)

adata.var["ribo"] = (
    gene_symbols
    .str.startswith(("RPS", "RPL"), na=False)
    .to_numpy(dtype=bool)
)

adata.var["hb"] = (
    gene_symbols
    .str.match(r"^HB[ABDEGQMZ]", na=False)
    .to_numpy(dtype=bool)
)


# Scanpy/SciPy require standard boolean masks rather than
# pandas BooleanArray or categorical columns.
for qc_var in ("mt", "ribo", "hb"):
    if adata.var[qc_var].dtype != bool:
        raise TypeError(
            f"adata.var[{qc_var!r}] has unexpected dtype: "
            f"{adata.var[qc_var].dtype}"
        )

    if adata.var[qc_var].isna().any():
        raise ValueError(
            f"adata.var[{qc_var!r}] contains missing values."
        )


# ------------------------------------------------------------------
# Calculate cell-level and gene-level QC metrics
# ------------------------------------------------------------------
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=[50, 100, 200, 500],
    inplace=True,
    log1p=True,
)


# ------------------------------------------------------------------
# Validate generated QC columns
# ------------------------------------------------------------------
required_obs_columns = [
    "n_genes_by_counts",
    "total_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
]

missing_obs_columns = [
    column
    for column in required_obs_columns
    if column not in adata.obs.columns
]

if missing_obs_columns:
    raise AssertionError(
        "QC metric calculation did not create the expected columns: "
        f"{missing_obs_columns}"
    )


# Force ordinary numeric dtypes to prevent later categorical-comparison
# errors in adaptive threshold calculations.
for column in required_obs_columns:
    adata.obs[column] = pd.to_numeric(
        adata.obs[column].astype("string"),
        errors="raise",
    ).astype("float64")

    if not np.isfinite(
        adata.obs[column].to_numpy(dtype=np.float64)
    ).all():
        raise ValueError(
            f"QC column {column!r} contains non-finite values."
        )


print("QC metrics calculated successfully.")
print("Mitochondrial genes:", int(adata.var["mt"].sum()))
print("Ribosomal genes:", int(adata.var["ribo"].sum()))
print("Hemoglobin genes:", int(adata.var["hb"].sum()))

adata.obs[required_obs_columns].head()

QC metrics calculated successfully.
Mitochondrial genes: 37
Ribosomal genes: 1128
Hemoglobin genes: 13


,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_ribo,pct_counts_hb
cell_id,,,,,
PJ052:TCCCCGGGAGTA,4298.0,9050.0,0.011050,10.320442,0.0
PJ052:TAATTGAAGACA,3929.0,8435.0,0.142264,15.483106,0.0
PJ052:GAAAGTGGTTCT,3752.0,7895.0,0.037999,12.615579,0.0
PJ052:CCCTAGTCACCG,3704.0,7086.0,0.225797,13.166808,0.0
PJ052:CGAGCTTGAGAC,3546.0,7071.0,0.098996,13.548296,0.0


In [6]:
import numpy as np
import pandas as pd

print("Running DS2 gene-symbol index patch v1")


# ------------------------------------------------------------------
# Validate required annotation
# ------------------------------------------------------------------
GENE_SYMBOL_COLUMN = "gene_symbols"

if GENE_SYMBOL_COLUMN not in adata.var.columns:
    raise KeyError(
        f"Missing adata.var[{GENE_SYMBOL_COLUMN!r}]. "
        f"Available columns: {adata.var.columns.tolist()}"
    )


# ------------------------------------------------------------------
# Preserve original versioned Ensembl identifiers
# ------------------------------------------------------------------
original_ensembl_ids = pd.Index(
    adata.var_names.astype(str),
    dtype="object",
)

if original_ensembl_ids.has_duplicates:
    raise AssertionError(
        "Original Ensembl feature identifiers are not unique."
    )

adata.var["ensembl_id_version"] = (
    original_ensembl_ids.to_numpy(dtype=object)
)


# ------------------------------------------------------------------
# Convert categorical gene symbols safely
# ------------------------------------------------------------------
symbol_series = (
    adata.var[GENE_SYMBOL_COLUMN]
    .astype("string")
    .fillna("")
    .str.strip()
)

symbol_values = symbol_series.to_numpy(
    dtype=object,
    na_value="",
)

symbol_values = np.asarray(
    [str(value).strip() for value in symbol_values],
    dtype=object,
)

invalid_tokens = {
    "",
    "nan",
    "none",
    "na",
    "<na>",
    "null",
}

invalid_symbol = np.asarray(
    [
        value.lower() in invalid_tokens
        for value in symbol_values
    ],
    dtype=bool,
)


# Use the original Ensembl identifier when a gene symbol is absent.
ensembl_values = original_ensembl_ids.to_numpy(dtype=object)

symbol_values[invalid_symbol] = (
    ensembl_values[invalid_symbol]
)


# ------------------------------------------------------------------
# Switch the feature index to gene symbols
# ------------------------------------------------------------------
adata.var_names = pd.Index(
    symbol_values.astype(str),
    dtype="object",
    name="gene_symbol",
)

if adata.var_names.isna().any():
    raise AssertionError(
        "New gene-symbol feature index contains missing values."
    )

if np.any(
    np.asarray(
        [str(value).strip() == "" for value in adata.var_names],
        dtype=bool,
    )
):
    raise AssertionError(
        "New gene-symbol feature index contains empty values."
    )

duplicated_symbols_before = int(
    adata.var_names.duplicated().sum()
)

adata.var_names_make_unique(join="-")

if adata.var_names.has_duplicates:
    raise AssertionError(
        "Gene-symbol feature identifiers remain duplicated."
    )

if adata.n_vars != len(original_ensembl_ids):
    raise AssertionError(
        "Feature count changed while replacing var_names."
    )


# ------------------------------------------------------------------
# Report completion
# ------------------------------------------------------------------
print("Feature axis switched to gene symbols successfully.")
print("Total features:", int(adata.n_vars))
print(
    "Missing/invalid symbols replaced with Ensembl IDs:",
    int(invalid_symbol.sum()),
)
print(
    "Duplicated symbols made unique:",
    duplicated_symbols_before,
)

adata.var[
    [GENE_SYMBOL_COLUMN, "ensembl_id_version"]
].head()

Running DS2 gene-symbol index patch v1
Feature axis switched to gene symbols successfully.
Total features: 60725
Missing/invalid symbols replaced with Ensembl IDs: 0
Duplicated symbols made unique: 1897


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_18732\1166480414.py:111: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['SNORD113-1', 'SNORD113-2', 'SNORD113-3', 'SNORD113-4', 'SNORD113-5']
  adata.var_names_make_unique(join="-")


,gene_symbols,ensembl_id_version
gene_symbol,,
Y_RNA,Y_RNA,ENSG00000201644.2
RP11-1072C15.4,RP11-1072C15.4,ENSG00000267160.1
BET1,BET1,ENSG00000105829.11
LINC01198,LINC01198,ENSG00000231817.7
ICE2,ICE2,ENSG00000128915.11


In [7]:
primary_rule_masks = pd.DataFrame(index=adata.obs_names)
primary_rule_masks["pass_min_genes"] = (
    adata.obs["n_genes_by_counts"] > MIN_GENES_EXCLUSIVE
)
primary_rule_masks["pass_max_genes"] = (
    adata.obs["n_genes_by_counts"] < MAX_GENES_EXCLUSIVE
)
primary_rule_masks["pass_min_counts"] = (
    adata.obs["total_counts"] > MIN_COUNTS_EXCLUSIVE
)
primary_rule_masks["pass_primary_max_counts"] = (
    adata.obs["total_counts"] < MAX_COUNTS_PRIMARY_EXCLUSIVE
)
primary_rule_masks["pass_max_mt"] = (
    adata.obs["pct_counts_mt"] < MAX_MT_PERCENT_EXCLUSIVE
)
primary_rule_masks["pass_primary_QC"] = primary_rule_masks.all(axis=1)

primary_rule_summary = pd.DataFrame(
    {
        "rule": primary_rule_masks.columns,
        "cells_passing": [
            int(primary_rule_masks[column].sum())
            for column in primary_rule_masks
        ],
        "cells_failing": [
            int((~primary_rule_masks[column]).sum())
            for column in primary_rule_masks
        ],
    }
)
primary_rule_summary.to_csv(
    AUDIT_DIR / "GSE141383_primary_QC_rule_summary.tsv",
    sep="\t",
    index=False,
)

adata_primary = adata[
    primary_rule_masks["pass_primary_QC"].to_numpy()
].copy()

if adata_primary.n_obs != EXPECTED_AFTER_PRIMARY_CELL_FILTER:
    raise AssertionError(
        "Primary DS2 cell filtering did not reproduce the original count. "
        f"Expected {EXPECTED_AFTER_PRIMARY_CELL_FILTER}, observed "
        f"{adata_primary.n_obs}."
    )

print("After primary cell filtering:", adata_primary.shape)
primary_rule_summary

After primary cell filtering: (21500, 60725)


,rule,cells_passing,cells_failing
0,pass_min_genes,25524,3891
1,pass_max_genes,29391,24
2,pass_min_counts,21566,7849
3,pass_primary_max_counts,29363,52
4,pass_max_mt,29342,73
5,pass_primary_QC,21500,7915


In [8]:
before_sample_counts = (
    adata.obs.groupby("sample", observed=True)
    .size()
    .rename("cells_before_QC")
)

final_cell_mask = (
    adata_primary.obs["total_counts"] <= FINAL_MAX_COUNTS_INCLUSIVE
)
adata_qc = adata_primary[final_cell_mask.to_numpy()].copy()

sc.pp.filter_genes(
    adata_qc,
    min_cells=FINAL_MIN_CELLS_PER_GENE,
)

after_sample_counts = (
    adata_qc.obs.groupby("sample", observed=True)
    .size()
    .rename("cells_after_QC")
)

sample_retention = pd.concat(
    [before_sample_counts, after_sample_counts],
    axis=1,
).fillna(0)
sample_retention["cells_removed"] = (
    sample_retention["cells_before_QC"]
    - sample_retention["cells_after_QC"]
)
sample_retention["retention_fraction"] = (
    sample_retention["cells_after_QC"]
    / sample_retention["cells_before_QC"]
)
sample_retention.to_csv(
    AUDIT_DIR / "GSE141383_QC_retention_by_sample.tsv",
    sep="\t",
)

if adata_qc.shape != EXPECTED_OUTPUT_SHAPE:
    raise AssertionError(
        "DS2 QC did not reproduce the recorded output shape. "
        f"Expected {EXPECTED_OUTPUT_SHAPE}, observed {adata_qc.shape}. "
        "Confirm that the same 29,415-cell merged object and compatible "
        "Scanpy/AnnData versions are being used."
    )

assert_raw_counts(adata_qc.X, "post-QC adata.X")
assert_raw_counts(
    adata_qc.layers["counts"],
    'post-QC adata.layers["counts"]',
)

print("Final post-QC object:", adata_qc)
sample_retention

Final post-QC object: AnnData object with n_obs × n_vars = 21391 × 25835
    obs: 'cell_barcode', 'sample', 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset', 'source_file', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'
    var: 'gene_symbols', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ensembl_id_version', 'n_cells'
    uns: 'build_timestamp', 'dataset_id', 'geo_accession', 'input_format', 'source_files'
    layers: 'counts'


,cells_before_QC,cells_after_QC,cells_removed,retention_fraction
sample,,,,
PDC001,7738,6325,1413,0.817395
PJ052,1839,1364,475,0.741707
PJ053,4459,3034,1425,0.680422
PW016-703,2811,1530,1281,0.544290
PW017-703,2831,1514,1317,0.534793
PW032-706,4747,4746,1,0.999789
PW032-710,992,979,13,0.986895
PW035-710,2205,953,1252,0.432200
PW039-705,1793,946,847,0.527607


In [9]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE141383_QC_metrics_before_filtering.png")

sc.pl.violin(
    adata_qc,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE141383_QC_metrics_after_filtering.png")

sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE141383_scatter_before_filtering.png")

sc.pl.scatter(
    adata_qc,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE141383_scatter_after_filtering.png")

In [10]:
qc_summary = {
    "dataset_id": "DS2_GSE141383",
    "geo_accession": "GSE141383",
    "input_cells": int(adata.n_obs),
    "input_features": int(adata.n_vars),
    "post_primary_filter_cells": int(adata_primary.n_obs),
    "post_QC_cells": int(adata_qc.n_obs),
    "post_QC_genes": int(adata_qc.n_vars),
    "retention_fraction": float(adata_qc.n_obs / adata.n_obs),
    "min_genes_exclusive": MIN_GENES_EXCLUSIVE,
    "max_genes_exclusive": MAX_GENES_EXCLUSIVE,
    "min_counts_exclusive": MIN_COUNTS_EXCLUSIVE,
    "primary_max_counts_exclusive": MAX_COUNTS_PRIMARY_EXCLUSIVE,
    "max_mt_percent_exclusive": MAX_MT_PERCENT_EXCLUSIVE,
    "final_max_counts_inclusive": FINAL_MAX_COUNTS_INCLUSIVE,
    "final_min_cells_per_gene": FINAL_MIN_CELLS_PER_GENE,
    "feature_axis": "gene symbols made unique",
    "original_feature_id_column": "ensembl_id_version",
    "raw_counts_in_X": True,
    "raw_counts_layer": "counts",
    "normalization_applied": False,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}

adata_qc.uns["dataset_level_QC"] = qc_summary

with open(
    AUDIT_DIR / "GSE141383_QC_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(qc_summary, handle, indent=2)

print(qc_summary)

{'dataset_id': 'DS2_GSE141383', 'geo_accession': 'GSE141383', 'input_cells': 29415, 'input_features': 60725, 'post_primary_filter_cells': 21500, 'post_QC_cells': 21391, 'post_QC_genes': 25835, 'retention_fraction': 0.7272140064592895, 'min_genes_exclusive': 300, 'max_genes_exclusive': 4500, 'min_counts_exclusive': 600, 'primary_max_counts_exclusive': 12000, 'max_mt_percent_exclusive': 3.0, 'final_max_counts_inclusive': 9000, 'final_min_cells_per_gene': 10, 'feature_axis': 'gene symbols made unique', 'original_feature_id_column': 'ensembl_id_version', 'raw_counts_in_X': True, 'raw_counts_layer': 'counts', 'normalization_applied': False, 'timestamp': '2026-08-03T15:52:37', 'output_h5ad': 'C:\\Users\\AmirSilco\\Documents\\GitHub\\glioma-cnv-single-cell-atlas\\data\\processed\\DS2_GSE141383\\GSE141383_postQC.h5ad'}


In [12]:
from pathlib import Path

import scanpy as sc


# ------------------------------------------------------------------
# Save the post-QC DS2 AnnData object atomically
# ------------------------------------------------------------------
OUTPUT_H5AD = Path(OUTPUT_H5AD)
OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)

temporary_output = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.temporary{OUTPUT_H5AD.suffix}"
)

# Remove a stale temporary file left by an interrupted prior run.
if temporary_output.exists():
    temporary_output.unlink()

adata_qc.write_h5ad(
    temporary_output,
    compression="gzip",
)

if not temporary_output.exists():
    raise FileNotFoundError(
        f"Temporary output was not created: {temporary_output}"
    )

# Atomic replacement of the final output.
temporary_output.replace(OUTPUT_H5AD)

if not OUTPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Final output was not created: {OUTPUT_H5AD}"
    )


# ------------------------------------------------------------------
# Reopen the saved file in backed mode and validate it
# ------------------------------------------------------------------
check = sc.read_h5ad(
    OUTPUT_H5AD,
    backed="r",
)

try:
    observed_shape = tuple(check.shape)
    expected_shape = tuple(EXPECTED_OUTPUT_SHAPE)

    if observed_shape != expected_shape:
        raise AssertionError(
            "Saved-file validation failed: "
            f"expected {expected_shape}, observed {observed_shape}"
        )

    if "counts" not in check.layers:
        raise AssertionError(
            "Saved output is missing layers['counts']."
        )

    if "ensembl_id_version" not in check.var.columns:
        raise AssertionError(
            "Saved output is missing "
            "var['ensembl_id_version']."
        )

    if check.obs_names.has_duplicates:
        raise AssertionError(
            "Saved output contains duplicated cell identifiers."
        )

    if check.var_names.has_duplicates:
        raise AssertionError(
            "Saved output contains duplicated feature identifiers."
        )

    if check.n_obs == 0:
        raise AssertionError(
            "Saved output contains zero cells."
        )

    if check.n_vars == 0:
        raise AssertionError(
            "Saved output contains zero features."
        )

finally:
    check.file.close()


# ------------------------------------------------------------------
# Report successful completion
# ------------------------------------------------------------------
print("Saved and validated:", OUTPUT_H5AD)
print("Validated shape:", expected_shape)
print("Validated layer: counts")
print("Validated var column: ensembl_id_version")

Saved and validated: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS2_GSE141383\GSE141383_postQC.h5ad
Validated shape: (21391, 25835)
Validated layer: counts
Validated var column: ensembl_id_version
